# Municipal Service Analytics — Weeks 1 and 2 (revised)

**One row = one service request.** This notebook profiles the raw dataset without editing it. Findings describe a capstone-program dataset, not a verified municipality.

Week 1: scope and business questions are in `README.md` and `docs/Week_1_Project_Proposal.md`. Week 2: dataset exploration and cross-field quality checks below.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
root = Path.cwd()
for base in [root, root.parent, root.parent.parent]:
    if (base / 'data/raw/Municipal_Service_Data.csv').exists():
        root = base
        break
else:
    raise FileNotFoundError('Run notebook from repository root or notebooks folder')
df = pd.read_csv(root / 'data/raw/Municipal_Service_Data.csv')
print('Shape:', df.shape)
display(df.head(), df.dtypes.to_frame('raw_type'))

Shape: (2000, 11)


,Service_Request_ID,Request_Type,Description,Department,Priority_Level,Status,Date_Opened,Date_Closed,Ward_District,Latitude,Longitude
0,SR00001,Garbage Collection,Service request related to municipal operations.,Public Works,High,Closed,2022-04-13 19:00:00,2022-04-28 08:00:00,South,49.233907,-123.020945
1,SR00002,Water Leakage,Service request related to municipal operations.,Public Works,Low,Open,2022-05-02 18:00:00,NaN,West,49.259853,-123.019665
2,SR00003,Pothole Repair,Service request related to municipal operations.,Public Works,Low,Closed,2023-10-26 02:00:00,NaN,South,49.216600,-122.918434
3,SR00004,Sewer Issue,Service request related to municipal operations.,Parks & Recreation,Low,Open,2023-01-21 20:00:00,2023-01-28 13:00:00,South,49.157427,-123.070416
4,SR00005,Sewer Issue,Service request related to municipal operations.,Electrical Services,Medium,In Progress,2023-07-15 09:00:00,2023-07-23 05:00:00,Central,49.136821,-122.964447


,raw_type
Service_Request_ID,object
Request_Type,object
Description,object
Department,object
Priority_Level,object
Status,object
Date_Opened,object
Date_Closed,object
Ward_District,object
Latitude,float64


## Completeness, duplicates and time fields

In [2]:
display(pd.DataFrame({'missing':df.isna().sum(), 'distinct_non_null':df.nunique(dropna=True)}))
print('Duplicate rows:', df.duplicated().sum())
print('Duplicate IDs:', df.Service_Request_ID.duplicated().sum())
opened = pd.to_datetime(df.Date_Opened, errors='coerce')
closed = pd.to_datetime(df.Date_Closed, errors='coerce')
print('Unparseable opening dates:', opened.isna().sum())
print('Unparseable non-empty closure dates:', (closed.isna() & df.Date_Closed.notna()).sum())
print('Opening range:', opened.min(), opened.max())
print('Closures before openings:', ((closed < opened) & closed.notna()).sum())
print('Distinct opening timestamps:', opened.nunique())

,missing,distinct_non_null
Service_Request_ID,0,2000
Request_Type,0,9
Description,0,1
Department,0,5
Priority_Level,0,4
Status,0,3
Date_Opened,0,1901
Date_Closed,185,1732
Ward_District,0,5
Latitude,0,1993


Duplicate rows: 0
Duplicate IDs: 0
Unparseable opening dates: 0
Unparseable non-empty closure dates: 0
Opening range: 2022-01-01 13:00:00 2023-12-31 15:00:00
Closures before openings: 0
Distinct opening timestamps: 1901


**Observation:** 185 closure dates are missing (9.25%). There are no duplicate IDs or duplicate full rows. `Date_Opened` has 1,901 distinct timestamps across 2,000 requests: some requests share an opening timestamp, which does **not** imply duplicate requests; it may reflect recording granularity or simultaneous entry.

## Distribution of categorical variables

In [3]:
for col in ['Request_Type','Department','Priority_Level','Status','Ward_District']:
    print('\n', col)
    display(df[col].value_counts(dropna=False).to_frame('count'))
print('Request type min/max:', df.Request_Type.value_counts().min(), df.Request_Type.value_counts().max())


 Request_Type


,count
Request_Type,
Noise Complaint,245
Road Damage,232
Water Leakage,228
Tree Maintenance,227
Sewer Issue,221
Pothole Repair,219
Street Light Outage,214
Garbage Collection,208
Graffiti Removal,206



 Department


,count
Department,
Electrical Services,420
Water Services,402
Parks & Recreation,401
Public Works,389
Sanitation,388



 Priority_Level


,count
Priority_Level,
Medium,524
Emergency,502
Low,498
High,476



 Status


,count
Status,
Closed,1198
In Progress,419
Open,383



 Ward_District


,count
Ward_District,
South,422
West,422
North,394
East,384
Central,378


Request type min/max: 206 245


**Observation:** The nine request categories are relatively evenly distributed: counts range from **206 to 245**. No single category overwhelmingly dominates. Staffing decisions should not rely on category volume alone; workload complexity and handling time are missing.

## Cross-field audit 1 — status vs. closure timestamp

In [4]:
missing_for_closed = ((df.Status == 'Closed') & closed.isna())
present_for_nonclosed = ((df.Status != 'Closed') & closed.notna())
print('Closed with missing close date:', missing_for_closed.sum())
print('Non-Closed with close date:', present_for_nonclosed.sum())
print('Total status/date conflicts:', (missing_for_closed | present_for_nonclosed).sum())
display(pd.crosstab(df.Status, closed.notna(), rownames=['Status'], colnames=['Has closure timestamp']))

Closed with missing close date: 98
Non-Closed with close date: 715
Total status/date conflicts: 813


Has closure timestamp,False,True
Status,,
Closed,98,1100
In Progress,44,375
Open,43,340


**Observation:** 98 Closed requests lack a closure timestamp, and 715 Open/In Progress requests have one: **813 contradictions (40.65%)**. Flag, do not silently change statuses or timestamps. These records affect valid resolution denominators.

## Cross-field audit 2 — request type vs. department

In [5]:
dept_by_type = pd.crosstab(df['Request_Type'], df['Department'])
display(dept_by_type)
print('Non-zero request-type/department combinations:', (dept_by_type>0).sum().sum(), 'out of', dept_by_type.size)
print('Min and max combination count:', dept_by_type.values.min(), dept_by_type.values.max())

Department,Electrical Services,Parks & Recreation,Public Works,Sanitation,Water Services
Request_Type,,,,,
Garbage Collection,44,38,40,41,45
Graffiti Removal,48,40,42,38,38
Noise Complaint,52,62,31,53,47
Pothole Repair,43,48,42,47,39
Road Damage,45,49,51,41,46
Sewer Issue,46,38,46,46,45
Street Light Outage,41,40,45,42,46
Tree Maintenance,51,35,43,51,47
Water Leakage,50,51,49,29,49


Non-zero request-type/department combinations: 45 out of 45
Min and max combination count: 29 62


**Observation:** All **45 of 45** request-type/department combinations occur (counts 29–62). For example, garbage collection appears under Electrical Services (44 records). This is a **plausibility flag, not proof of misassignment**, because the actual routing rules were not supplied. Department-level comparisons need a validated service-to-department mapping.

## Cross-field audit 3 — district vs. coordinates

In [6]:
district_bounds = df.groupby('Ward_District')[['Latitude','Longitude']].agg(['min','max'])
display(district_bounds)
print('Overall latitude:',df.Latitude.min(),df.Latitude.max())
print('Overall longitude:',df.Longitude.min(),df.Longitude.max())

Latitude              Longitude            
                     min        max         min         max
Ward_District                                              
Central        49.001351  49.298970 -123.199187 -122.900085
East           49.000237  49.299322 -123.199402 -122.900860
North          49.000215  49.299801 -123.199436 -122.900845
South          49.001111  49.299852 -123.198824 -122.900541
West           49.000071  49.299151 -123.199717 -122.900330

Overall latitude: 49.000071 49.299852
Overall longitude: -123.199717 -122.900085


**Observation:** Each district spans nearly the same latitude/longitude range (approximately 49.00–49.30 latitude and −123.20 to −122.90 longitude). Ranges strongly overlap. This does **not prove** coordinates are wrong: without actual district polygons, membership cannot be validated. A spatial boundary file is required before mapping district accuracy.

## Week 2 conclusions and Week 3 handoff

- Data quality is a major constraint: 813 status/date contradictions; retain flags and clear denominators.
- Request categories are unusually balanced and routing patterns are broad; seek operational validation.
- Geographic district labels are not independently verifiable from the supplied coordinate ranges.
- Do not delete the 185 missing closure dates or impute closure status without business rules.
- Week 3 will parse dates, standardize categories, add quality flags and create a reproducible processed dataset while preserving raw records.
- See `docs/Data_Dictionary.md` for all 11 raw columns.